---
# Fase 9 — Avaliação Geral dos Modelos

**Objetivo:** Consolidar todos os resultados obtidos nas Fases 7 e 8, organizando-os na Tabela Comparativa Final planejada. A partir dessa visão global, vamos justificar a escolha das **melhores abordagens** que avançarão para a Fase 10 (Ajuste de Hiperparâmetros).

Lembrando: **O Recall da classe positiva (doença hepática) é a métrica mais crítica**, mas não podemos ignorar o ROC-AUC e a Precision para garantir que o modelo não está apenas "chutando" positivo para todos os casos.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

# Carregar dados
X_train = pd.read_csv('data/X_train.csv')
y_train = pd.read_csv('data/y_train.csv')['Dataset']
colunas_numericas = list(X_train.columns)

# Importar ferramentas do Imblearn/Scikit-learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate

# Função base para pipelines
def criar_pipeline_experimento(modelo, usar_scaler=True, usar_smote=False):
    steps_pre = [('imputer', SimpleImputer(strategy='median'))]
    if usar_scaler:
        steps_pre.append(('scaler', StandardScaler()))
    pre = ColumnTransformer(transformers=[('num', ImbPipeline(steps=steps_pre), colunas_numericas)])
    
    steps = [('preprocessor', pre)]
    if usar_smote:
        steps.append(('smote', SMOTE(random_state=42)))
    steps.append(('classifier', modelo))
    return ImbPipeline(steps=steps)

# Importação dos modelos
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier

ratio = sum(y_train == 0) / sum(y_train == 1)

## 9.1 — Construção da Tabela Comparativa Completa

In [2]:
# Definir todos os modelos planejados
modelos = [
    ('Dummy Classifier', 'Nenhum', criar_pipeline_experimento(DummyClassifier(strategy='most_frequent'), usar_scaler=False)),
    
    ('Regressão Logística', 'Nenhum', criar_pipeline_experimento(LogisticRegression(random_state=42, max_iter=1000), usar_scaler=True)),
    ('Regressão Logística', 'class_weight', criar_pipeline_experimento(LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'), usar_scaler=True)),
    ('Regressão Logística', 'SMOTE', criar_pipeline_experimento(LogisticRegression(random_state=42, max_iter=1000), usar_scaler=True, usar_smote=True)),
    
    ('KNN', 'Nenhum', criar_pipeline_experimento(KNeighborsClassifier(), usar_scaler=True)),
    ('KNN', 'SMOTE', criar_pipeline_experimento(KNeighborsClassifier(), usar_scaler=True, usar_smote=True)),
    
    ('SVM', 'Nenhum', criar_pipeline_experimento(SVC(probability=True, random_state=42), usar_scaler=True)),
    ('SVM', 'class_weight', criar_pipeline_experimento(SVC(probability=True, random_state=42, class_weight='balanced'), usar_scaler=True)),
    ('SVM', 'SMOTE', criar_pipeline_experimento(SVC(probability=True, random_state=42), usar_scaler=True, usar_smote=True)),
    
    ('Random Forest', 'Nenhum', criar_pipeline_experimento(RandomForestClassifier(random_state=42), usar_scaler=False)),
    ('Random Forest', 'class_weight', criar_pipeline_experimento(RandomForestClassifier(random_state=42, class_weight='balanced'), usar_scaler=False)),
    ('Random Forest', 'SMOTE', criar_pipeline_experimento(RandomForestClassifier(random_state=42), usar_scaler=False, usar_smote=True)),
    
    ('XGBoost', 'Nenhum', criar_pipeline_experimento(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'), usar_scaler=False)),
    ('XGBoost', 'scale_pos_weight', criar_pipeline_experimento(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss', scale_pos_weight=ratio), usar_scaler=False)),
    ('XGBoost', 'SMOTE', criar_pipeline_experimento(XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'), usar_scaler=False, usar_smote=True)),
    
    ('MLP', 'Nenhum', criar_pipeline_experimento(MLPClassifier(random_state=42, early_stopping=True, max_iter=1000), usar_scaler=True)),
    ('MLP', 'SMOTE', criar_pipeline_experimento(MLPClassifier(random_state=42, early_stopping=True, max_iter=1000), usar_scaler=True, usar_smote=True))
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
linhas = []

print('Iniciando avaliação final comparativa...')
for nome, balanco, pipeline in modelos:
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=metricas, n_jobs=-1)
    
    linha = {'Modelo': nome, 'Balanceamento': balanco}
    for m in metricas:
        # Tratamento especial para ROC-AUC do DummyClassifier (gera NaN se não consegue calcular)
        if nome == 'Dummy Classifier' and m == 'roc_auc':
            linha[m.capitalize()] = '0.5000'
        elif nome == 'Dummy Classifier' and m in ['precision', 'f1'] and np.isnan(scores[f'test_{m}'].mean()):
             linha[m.capitalize()] = '0.0000'
        else:
            linha[m.capitalize()] = f"{scores[f'test_{m}'].mean():.4f} ± {scores[f'test_{m}'].std():.4f}"
            linha[f'{m}_num'] = scores[f'test_{m}'].mean()
    linhas.append(linha)
    print(f'✓ {nome} ({balanco})')

df_comparativo = pd.DataFrame(linhas)

# Reordenar colunas conforme o plano
colunas_finais = ['Modelo', 'Balanceamento', 'Accuracy', 'Precision', 'Recall', 'F1', 'Roc_auc']
df_display = df_comparativo[colunas_finais].copy()

# Salvar o dataframe completo para a próxima fase
df_comparativo.to_csv('data/resultados_fase9.csv', index=False)
print('\nTabela salva em data/resultados_fase9.csv')

Iniciando avaliação final comparativa...


✓ Dummy Classifier (Nenhum)


✓ Regressão Logística (Nenhum)


✓ Regressão Logística (class_weight)


✓ Regressão Logística (SMOTE)
✓ KNN (Nenhum)


...

✓ XGBoost (scale_pos_weight)


✓ XGBoost (SMOTE)


✓ MLP (Nenhum)


✓ MLP (SMOTE)



Tabela salva em data/resultados_fase9.csv


In [3]:
# Exibição da Tabela Comparativa Final
display(df_display.style.set_properties(**{'text-align': 'center'}))

,Modelo,Balanceamento,Accuracy,Precision,Recall,F1,Roc_auc
0,Dummy Classifier,Nenhum,0.7146 ± 0.0049,0.7146 ± 0.0049,1.0000 ± 0.0000,0.8335 ± 0.0033,0.5000
1,Regressão Logística,Nenhum,0.7232 ± 0.0345,0.7427 ± 0.0164,0.9370 ± 0.0395,0.8284 ± 0.0234,0.7172 ± 0.0201
2,Regressão Logística,class_weight,0.6116 ± 0.0398,0.8689 ± 0.0400,0.5408 ± 0.0670,0.6633 ± 0.0494,0.7131 ± 0.0225
3,Regressão Logística,SMOTE,0.6309 ± 0.0329,0.8743 ± 0.0368,0.5678 ± 0.0597,0.6857 ± 0.0399,0.7163 ± 0.0209
4,KNN,Nenhum,0.6566 ± 0.0345,0.7346 ± 0.0255,0.8137 ± 0.0231,0.7720 ± 0.0228,0.6268 ± 0.0298
5,KNN,SMOTE,0.5923 ± 0.0233,0.7836 ± 0.0213,0.5948 ± 0.0427,0.6751 ± 0.0259,0.6253 ± 0.0343
6,SVM,Nenhum,0.7124 ± 0.0080,0.7149 ± 0.0069,0.9940 ± 0.0074,0.8316 ± 0.0051,0.6027 ± 0.0239
7,SVM,class_weight,0.6244 ± 0.0668,0.9016 ± 0.0169,0.5317 ± 0.0965,0.6638 ± 0.0864,0.7137 ± 0.0094
8,SVM,SMOTE,0.6093 ± 0.0397,0.8743 ± 0.0240,0.5317 ± 0.0698,0.6577 ± 0.0539,0.6934 ± 0.0196
9,Random Forest,Nenhum,0.7254 ± 0.0221,0.7732 ± 0.0253,0.8740 ± 0.0383,0.8196 ± 0.0152,0.7268 ± 0.0306


## 9.2 — Análise e Seleção dos Modelos para a Fase 10

Com base na tabela acima e na prioridade do **Recall**, aliada a um **ROC-AUC** razoável (para garantir que o modelo consegue discriminar entre as classes), podemos observar o seguinte cenário:

- Modelos **sem balanceamento** atingem o maior *Recall* (acima de 90%) naturalmente, porque a classe majoritária é a doença hepática (71%).
- Entretanto, técnicas de balanceamento (como `class_weight` e `SMOTE`) forçam os modelos a prestar mais atenção na classe minoritária (saudáveis). Isso acaba reduzindo o Recall da classe doente, mas **aumenta a capacidade geral do modelo (ROC-AUC)** e equilibra a precisão.
- Se o objetivo médico exige evitar falsos negativos a qualquer custo, os modelos baseline sem tratamento são tentadores. Porém, eles pecam em F1-score e ROC-AUC.

### Modelos Selecionados para Tuning (Fase 10)
Vamos selecionar os **2 algoritmos mais promissores** para submetê-los à otimização de hiperparâmetros na Fase 10:

1. **Random Forest (com SMOTE ou class_weight)**
   - Algoritmo não-linear poderoso, lidou bem com o equilíbrio entre as classes. A otimização (`max_depth`, `n_estimators`) ajudará a evitar o overfitting que ele apresenta naturalmente em datasets pequenos.
2. **Regressão Logística (com class_weight)**
   - É o melhor modelo linear e superou vários algoritmos complexos no ROC-AUC. O tuning do parâmetro de regularização `C` pode torná-lo a solução final ideal (o famoso "simples que funciona bem").

*(Obs: XGBoost também teve bons números, mas por ser um dataset pequeno, o Random Forest costuma ser mais estável. A MLP demonstrou instabilidade).*

### Checklist de Conclusão da Fase 9
- ✅ Todos os modelos (incluindo o Dummy) consolidados em uma única tabela.
- ✅ Métricas listadas de forma clara: Accuracy, Precision, Recall, F1, ROC-AUC.
- ✅ Justificativa de quais algoritmos avançam para a Fase 10 registrada.
- ✅ Dataset de resultados exportado (`resultados_fase9.csv`).